# Summary_Day10_online.ipynb  
## 다중 분류 2 · 인터넷 가능 버전 · MNIST 다운로드 기반 MLP 이미지 분류

이번 10강은 9강에서 배운 다중 분류를 **이미지 데이터**에 적용하는 강의다.

강사님은 오늘 수업을 CNN으로 넘어가기 전, 기본 MLP 딥러닝 구조를 정리하는 과정이라고 설명했다.  
핵심은 MNIST 손글씨 숫자 이미지를 이용해 다음 흐름을 이해하는 것이다.

```text
이미지 = 픽셀 숫자 격자
MNIST = 28 × 28 흑백 이미지
28 × 28 = 784개 픽셀 feature
PyTorch 이미지 순서 = CHW
완전 연결 신경망 = 1차원 벡터 입력 필요
Transform = ToTensor + Normalize + Flatten
DataLoader = mini-batch 단위로 데이터 공급
MLP = Linear → ReLU → Linear
학습 = Forward → Loss → Backward → Optimizer Step
```

이 파일은 **인터넷이 되는 환경**을 기준으로 한다.  
`torchvision.datasets.MNIST(download=True)`로 MNIST를 직접 내려받아 실습한다.

> 필기 포인트:  
> 사람은 숫자를 모양으로 본다.  
> 컴퓨터는 이미지를 의미로 보는 것이 아니라, 0~255 또는 0~1 사이 픽셀 숫자의 격자로 본다.

## 1. 인터넷 가능 버전의 전체 목적

이 노트북의 목적은 다음이다.

1. MNIST 이미지를 직접 다운로드한다.
2. 이미지가 `[1, 28, 28]` 형태로 들어오는 것을 확인한다.
3. 완전 연결 신경망에 넣기 위해 이미지를 `[784]`로 펼친다.
4. `DataLoader`로 mini-batch 학습 구조를 만든다.
5. MLP 모델로 숫자 0~9를 다중 분류한다.
6. `CrossEntropyLoss`와 `torch.max(outputs, 1)[1]` 패턴을 다시 익힌다.
7. GPU가 있으면 GPU를 사용하고, 없으면 CPU로 실행한다.

## 2. 라이브러리 준비

### 함수/모듈 사용법

```python
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
```

- `torch`: Tensor와 자동 미분을 사용한다.
- `nn`: 신경망 Layer와 손실함수를 만든다.
- `optim`: Optimizer를 만든다.
- `datasets.MNIST`: MNIST 데이터를 내려받고 불러온다.
- `transforms`: 이미지 전처리를 연결한다.
- `DataLoader`: 데이터를 mini-batch 단위로 꺼낸다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("PyTorch:", torch.__version__)

## 3. GPU / CPU device 설정

강사님은 CPU와 GPU가 서로 다른 장치라서, 모델과 Tensor가 같은 device에 있어야 한다고 강조했다.

### 함수 사용법

```python
torch.cuda.is_available()
torch.device("cuda" if torch.cuda.is_available() else "cpu")
tensor.to(device)
model.to(device)
```

- `torch.cuda.is_available()`: CUDA GPU 사용 가능 여부를 확인한다.
- `.to(device)`: Tensor나 모델을 CPU/GPU로 이동한다.
- 모델은 GPU, 데이터는 CPU에 있으면 연산 에러가 난다.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("사용 device:", device)

## 4. MNIST 이미지 구조 이해

MNIST는 손글씨 숫자 0~9 이미지 데이터다.

```text
이미지 크기: 28 × 28
채널 수: 1개, 흑백
class 수: 10개, 0~9
픽셀 수: 28 × 28 = 784개
```

PyTorch에서 이미지 Tensor는 보통 다음 순서다.

```text
C, H, W
channel, height, width
```

MNIST 한 장은 다음 형태가 된다.

```text
[1, 28, 28]
```

> 시험 포인트:  
> NumPy나 일반 이미지 라이브러리는 HWC 순서를 자주 쓰고, PyTorch는 CHW 순서를 자주 쓴다.

## 5. Transform 정의하기

강의에서 나온 전처리 흐름은 다음과 같다.

```text
ToTensor → Normalize → Flatten
```

### 함수 사용법

```python
transforms.Compose([...])
```

- 여러 전처리 단계를 순서대로 묶는다.

```python
transforms.ToTensor()
```

- 이미지를 Tensor로 바꾼다.
- 픽셀 범위를 0~255에서 0~1로 바꾼다.

```python
transforms.Normalize((0.5,), (0.5,))
```

- 평균 0.5, 표준편차 0.5 기준으로 정규화한다.
- 0~1 범위를 대략 -1~1 범위로 바꾼다.

```python
transforms.Lambda(lambda x: x.view(-1))
```

- `[1, 28, 28]` 이미지를 `[784]` 벡터로 펼친다.
- 완전 연결 신경망은 1차원 feature vector를 입력으로 받기 때문이다.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    transforms.Lambda(lambda x: x.view(-1))
])

print(transform)

## 6. MNIST 데이터 다운로드

인터넷이 되는 환경에서는 `download=True`로 MNIST를 내려받을 수 있다.

### 함수 사용법

```python
datasets.MNIST(root="./data", train=True, download=True, transform=transform)
```

- `root`: 데이터 저장 폴더다.
- `train=True`: 훈련 데이터를 불러온다.
- `train=False`: 테스트 데이터를 불러온다.
- `download=True`: 없으면 인터넷에서 다운로드한다.
- `transform`: 이미지를 불러올 때 적용할 전처리다.

In [ ]:
train_dataset_full = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset_full = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("train 데이터 개수:", len(train_dataset_full))
print("test 데이터 개수:", len(test_dataset_full))

## 7. 빠른 실습을 위한 Subset 만들기

MNIST 전체는 train 60,000개, test 10,000개다.

강의 전체 흐름을 빠르게 확인하려면 일부만 사용해도 된다.

### 함수 사용법

```python
Subset(dataset, indices)
```

- 큰 dataset에서 지정한 index만 골라 작은 dataset을 만든다.
- 실습 속도를 줄일 때 유용하다.

In [ ]:
train_indices = list(range(10000))
test_indices = list(range(2000))

train_dataset = Subset(train_dataset_full, train_indices)
test_dataset = Subset(test_dataset_full, test_indices)

print("실습용 train 개수:", len(train_dataset))
print("실습용 test 개수:", len(test_dataset))

## 8. DataLoader 만들기

DataLoader는 데이터를 mini-batch 단위로 꺼내주는 도구다.

강사님은 60,000개를 500개씩 묶으면 120박스가 되는 식으로 설명했다.

### 함수 사용법

```python
DataLoader(dataset, batch_size=500, shuffle=True)
```

- `dataset`: 사용할 데이터셋이다.
- `batch_size`: 한 번에 꺼낼 데이터 개수다.
- `shuffle=True`: 학습 데이터 순서를 epoch마다 섞는다.
- 검증/테스트 데이터는 보통 `shuffle=False`로 둔다.

In [ ]:
batch_size = 500

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("train batch 개수:", len(train_loader))
print("test batch 개수:", len(test_loader))

## 9. Batch shape 확인하기

DataLoader에서 mini-batch 하나를 꺼내 shape을 확인한다.

전처리에서 이미 Flatten을 했기 때문에 이미지 batch는 `[batch_size, 784]` 형태가 된다.

In [ ]:
images, labels = next(iter(train_loader))

print("images shape:", images.shape)
print("labels shape:", labels.shape)
print("images dtype:", images.dtype)
print("labels dtype:", labels.dtype)
print("첫 번째 label:", labels[0].item())

출력 해석:

```text
images shape = [500, 784]
labels shape = [500]
```

- 500은 mini-batch 크기다.
- 784는 28×28 이미지를 펼친 feature 개수다.
- label은 0~9 정수 class다.
- `CrossEntropyLoss`를 쓰므로 label dtype은 `torch.long`이어야 한다.

## 10. 이미지 다시 보기

Flatten된 `[784]` 벡터를 다시 `[28, 28]`로 바꾸면 사람 눈으로 볼 수 있다.

### 함수 사용법

```python
image.view(28, 28)
```

- 784개 벡터를 28×28 행렬로 되돌린다.
- 시각화할 때 사용한다.

In [ ]:
sample_img = images[0].view(28, 28)

plt.imshow(sample_img, cmap="gray")
plt.title(f"label = {labels[0].item()}")
plt.axis("off")
plt.show()

## 11. MLP 모델 정의

완전 연결 신경망 MLP는 이미지를 벡터로 펼친 뒤 Linear Layer에 넣는다.

구조는 다음이다.

```text
입력 784개
→ Linear(784, 128)
→ ReLU
→ Linear(128, 10)
```

- 784는 픽셀 feature 개수다.
- 128은 은닉층 뉴런 수다.
- 10은 숫자 class 0~9 개수다.

In [ ]:
class MNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

model = MNISTMLP().to(device)

print(model)

total_params = sum(p.numel() for p in model.parameters())
print("전체 parameter 수:", total_params)

### 함수 사용법 정리

```python
nn.Linear(784, 128)
```

- 784개 입력 feature를 128개 은닉 표현으로 바꾼다.

```python
nn.ReLU()
```

- 음수는 0, 양수는 그대로 통과시킨다.
- 은닉층에 비선형성을 넣는다.

```python
nn.Linear(128, 10)
```

- 10개 class에 대한 logits를 출력한다.
- 마지막에 Softmax를 붙이지 않는다.

## 12. 손실함수와 Optimizer

다중 분류의 표준 손실함수는 `CrossEntropyLoss`다.

### 함수 사용법

```python
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
```

- `CrossEntropyLoss`는 logits와 정수 label을 입력으로 받는다.
- `Adam`은 자주 쓰는 Optimizer다.
- `model.parameters()`는 모델의 weight와 bias를 Optimizer에게 넘긴다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("criterion:", criterion)
print("optimizer:", optimizer.__class__.__name__)

## 13. 한 batch로 Forward 확인

모델 출력 shape을 확인한다.

다중 분류 출력은 `[batch_size, num_classes]`다.

In [ ]:
images_device = images.to(device)
labels_device = labels.to(device)

outputs = model(images_device)
loss = criterion(outputs, labels_device)

print("outputs shape:", outputs.shape)
print("loss:", loss.item())

pred = torch.max(outputs, 1)[1]
print("pred shape:", pred.shape)
print("첫 10개 예측:", pred[:10].detach().cpu().numpy())

출력 해석:

```text
outputs shape = [500, 10]
```

- 500개 이미지 각각에 대해 10개 숫자 class 점수를 출력한다.
- 최종 예측 class는 `torch.max(outputs, 1)[1]`로 구한다.

## 14. 학습 함수 만들기

한 epoch 학습 흐름은 다음이다.

```text
model.train()
→ batch를 device로 이동
→ optimizer.zero_grad()
→ outputs = model(images)
→ loss = criterion(outputs, labels)
→ loss.backward()
→ optimizer.step()
```

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pred = torch.max(outputs, 1)[1]
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy

## 15. 평가 함수 만들기

평가할 때는 gradient 계산이 필요 없다.

### 함수 사용법

```python
model.eval()
with torch.no_grad():
```

- `model.eval()`: 평가 모드로 바꾼다.
- `torch.no_grad()`: gradient 계산을 끈다.
- 평가 속도가 빨라지고 메모리 사용량이 줄어든다.

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            pred = torch.max(outputs, 1)[1]

            correct += (pred == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(pred.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy, np.array(all_labels), np.array(all_preds)

## 16. 모델 학습 실행

실습용 subset을 사용하므로 몇 epoch만 돌려도 흐름을 확인할 수 있다.

In [ ]:
num_epochs = 3

history = {
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_acc": []
}

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, _, _ = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(
        f"epoch {epoch + 1} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"test_loss={test_loss:.4f} | test_acc={test_acc:.4f}"
    )

## 17. 학습 곡선 확인

Loss와 Accuracy를 그래프로 확인한다.

In [ ]:
plt.plot(history["train_loss"], label="train loss")
plt.plot(history["test_loss"], label="test loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("MNIST Online Loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(history["train_acc"], label="train acc")
plt.plot(history["test_acc"], label="test acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("MNIST Online Accuracy")
plt.legend()
plt.show()

그래프 해석:

- loss가 내려가면 모델이 정답 class에 더 높은 점수를 주고 있다는 뜻이다.
- accuracy가 올라가면 숫자 class를 더 많이 맞힌다는 뜻이다.
- train과 test가 비슷하게 좋아지면 과적합이 심하지 않은 상태다.

## 18. Classification Report와 Confusion Matrix

숫자별 성능을 확인한다.

In [ ]:
test_loss, test_acc, y_true, y_pred = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print("test accuracy:", test_acc)

target_names = [f"digit {i}" for i in range(10)]

print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
print(cm)

In [ ]:
plt.imshow(cm)
plt.title("MNIST Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

for (i, j), value in np.ndenumerate(cm):
    if value > 0:
        plt.text(j, i, str(value), ha="center", va="center", fontsize=8)

plt.colorbar()
plt.show()

그래프 해석:

- 대각선은 맞힌 숫자 개수다.
- 대각선 밖의 값은 모델이 헷갈린 숫자다.
- 예를 들어 4를 9로 예측했다면 4행 9열에 값이 생긴다.

## 19. Softmax로 개별 예측 확률 보기

`CrossEntropyLoss`를 사용했기 때문에 모델 출력은 logits다.  
사람이 확률처럼 해석하려면 Softmax를 적용한다.

In [ ]:
model.eval()

images_batch, labels_batch = next(iter(test_loader))
images_batch = images_batch.to(device)

with torch.no_grad():
    logits = model(images_batch[:5])
    probs = torch.softmax(logits, dim=1)
    preds = torch.max(logits, 1)[1]

for i in range(5):
    print("sample", i)
    print("true:", labels_batch[i].item())
    print("pred:", preds[i].item())
    print("prob:", probs[i].cpu().numpy().round(3))
    print()

## 20. 4차원 이미지 Tensor로 다시 생각하기

완전 연결 MLP는 이미지를 `[784]`로 펼쳐서 사용한다.

하지만 CNN에서는 이미지를 펼치지 않고 다음 구조로 사용한다.

```text
[batch, channel, height, width]
```

MNIST batch는 CNN 기준으로 다음 형태가 된다.

```text
[batch, 1, 28, 28]
```

> 다음 CNN 강의로 넘어가면 Flatten 대신 이미지의 공간 구조를 유지하는 방식이 중요해진다.

In [ ]:
raw_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

raw_train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=raw_transform
)

raw_loader = DataLoader(raw_train_dataset, batch_size=32, shuffle=True)

raw_images, raw_labels = next(iter(raw_loader))

print("CNN용 raw image batch shape:", raw_images.shape)
print("label shape:", raw_labels.shape)

## 21. 핵심 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 사용법 |
|---|---|---|
| `MNIST` | 손글씨 숫자 이미지 데이터 | `datasets.MNIST(...)` |
| `pixel` | 이미지 한 칸의 숫자값 | 0~255 또는 0~1 |
| `CHW` | PyTorch 이미지 순서 | channel, height, width |
| `HWC` | 일반 이미지 순서 | height, width, channel |
| `ToTensor` | 이미지 Tensor 변환 | 0~1 범위로 변환 |
| `Normalize` | 정규화 | 평균/표준편차 기준 변환 |
| `Flatten` | 1차원 벡터화 | `[1,28,28] → [784]` |
| `DataLoader` | mini-batch 생성 | `DataLoader(dataset, batch_size=...)` |
| `batch_size` | 한 번에 학습할 데이터 수 | 500 등 |
| `shuffle` | 데이터 순서 섞기 | train은 True, test는 False |
| `MLP` | 완전 연결 신경망 | Linear 중심 구조 |
| `ReLU` | 활성화 함수 | 음수는 0, 양수는 그대로 |
| `CrossEntropyLoss` | 다중 분류 손실 | logits + long label |
| `torch.max(outputs, 1)[1]` | 예측 class 추출 | indices가 예측 label |
| `device` | CPU/GPU 위치 | `cuda` 또는 `cpu` |

## 22. 시험용 요약

```text
MNIST MLP = 28×28 이미지를 784개 벡터로 펼쳐 0~9 숫자를 분류하는 모델
```

꼭 기억할 것:

- MNIST는 28×28 흑백 손글씨 숫자 데이터다.
- 컴퓨터는 이미지를 픽셀 숫자 격자로 본다.
- PyTorch 이미지는 보통 CHW 순서를 쓴다.
- MNIST 한 장은 `[1, 28, 28]`이다.
- 완전 연결 신경망은 1차원 입력을 받으므로 Flatten이 필요하다.
- `28 × 28 = 784`다.
- `ToTensor()`는 이미지를 Tensor로 바꾸고 0~1 범위로 만든다.
- `Normalize((0.5,), (0.5,))`는 대략 -1~1 범위로 바꾼다.
- `DataLoader`는 데이터를 mini-batch 단위로 묶는다.
- train loader는 보통 `shuffle=True`다.
- test loader는 보통 `shuffle=False`다.
- 모델과 데이터는 같은 device에 있어야 한다.
- MLP 구조는 `Linear → ReLU → Linear`다.
- 다중 분류에서는 마지막 출력이 class 수만큼 나온다.
- MNIST는 class가 10개라 출력도 10개다.
- `CrossEntropyLoss`에는 Softmax를 먼저 붙이지 않는다.
- 예측 class는 `torch.max(outputs, 1)[1]`로 구한다.
- 평가할 때는 `model.eval()`과 `torch.no_grad()`를 사용한다.